In [1]:
from pathlib import Path
dir_path = "../data/fastapi/docs"
def collect_files(dir_path: str)-> list[str]:
    """Collects all file paths into a list"""
    doc_path = Path(dir_path).resolve()
    if not doc_path.exists():
        raise FileNotFoundError(f"Path does not exist: {doc_path}")
    
    file_paths = [str(f) for f in sorted(doc_path.rglob("*.md"))]
    print(f"[Collect] Found {len(file_paths)} markdown files in {dir_path}")
    return file_paths

print(collect_files(dir_path))

[Collect] Found 153 markdown files in ../data/fastapi/docs
['C:\\Abhyuday Drive\\Learnings\\Projects on GenAI Engineering\\Projects\\AI-powered documentation assistant\\data\\fastapi\\docs\\_llm-test.md', 'C:\\Abhyuday Drive\\Learnings\\Projects on GenAI Engineering\\Projects\\AI-powered documentation assistant\\data\\fastapi\\docs\\about\\index.md', 'C:\\Abhyuday Drive\\Learnings\\Projects on GenAI Engineering\\Projects\\AI-powered documentation assistant\\data\\fastapi\\docs\\advanced\\additional-responses.md', 'C:\\Abhyuday Drive\\Learnings\\Projects on GenAI Engineering\\Projects\\AI-powered documentation assistant\\data\\fastapi\\docs\\advanced\\additional-status-codes.md', 'C:\\Abhyuday Drive\\Learnings\\Projects on GenAI Engineering\\Projects\\AI-powered documentation assistant\\data\\fastapi\\docs\\advanced\\advanced-dependencies.md', 'C:\\Abhyuday Drive\\Learnings\\Projects on GenAI Engineering\\Projects\\AI-powered documentation assistant\\data\\fastapi\\docs\\advanced\\advan

In [2]:
def parse_file(filepath: str) -> dict:
    """Read a markdown file and return its content with metadata"""
    path = Path(filepath)
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        content = f.read()
        return {
            "source": path.name,
            "path": str(path),
            "content": content,
        }
file_paths = collect_files(dir_path)
document = parse_file(file_paths[0])
print(document)

[Collect] Found 153 markdown files in ../data/fastapi/docs
{'source': '_llm-test.md', 'path': 'C:\\Abhyuday Drive\\Learnings\\Projects on GenAI Engineering\\Projects\\AI-powered documentation assistant\\data\\fastapi\\docs\\_llm-test.md', 'content': '# LLM test file { #llm-test-file }\n\nThis document tests if the <abbr title="Large Language Model">LLM</abbr>, which translates the documentation, understands the `general_prompt` in `scripts/translate.py` and the language specific prompt in `docs/{language code}/llm-prompt.md`. The language specific prompt is appended to `general_prompt`.\n\nTests added here will be seen by all designers of language specific prompts.\n\nUse as follows:\n\n* Have a language specific prompt - `docs/{language code}/llm-prompt.md`.\n* Do a fresh translation of this document into your desired target language (see e.g. the `translate-page` command of the `translate.py`). This will create the translation under `docs/{language code}/docs/_llm-test.md`.\n* Check 

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import hashlib

def chunk_file(filepath: str) -> list[dict]:
    """Parse + Chunk a single Markdown File"""

    # Parsing 
    doc = parse_file(filepath)

    if not doc["content"].strip():
        return []
    
    md_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "h1"),
            ("##", "h2"),
            ("###", "h3"),
        ],
        strip_headers=False,
        
    )
    md_sections = md_splitter.split_text(doc["content"])
    print("Number of splits in current file" , md_sections)

    char_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1500,
        chunk_overlap = 200,
        separators=[
            "\n```\n",
            "\n\n"
            "\n"
            " "
        ],
    )

    final_chunks = char_splitter.split_documents(md_sections)
    print("final number of chunks in current file : ", final_chunks)


    # Build Output
    results = []
    for i, chunk in enumerate(final_chunks):
        text = chunk.page_content.strip()
        if not text:
            continue

        headers = chunk.metadata
        heading_trail = ">".join(headers[k] for k in ("h1", "h2", "h3") if k in headers)

        raw_id = f"{doc['source']}::{i}::{text[:100]}"
        chunk_id = hashlib.md5(raw_id.encode()).hexdigest()
        results.append({
            "chunk_id": chunk_id,
            "text": text,
            "source": doc["source"],
            "heading_trail": heading_trail
        })

    return results
    


print(chunk_file(file_paths[0]))

In [ ]:
from concurrent.futures import ProcessPoolExecutor

# Chunking in batches 

def chunk_batchs_in_multiprocess(file_paths: list[str], max_workers:int = None) -> list[dict]:
    """
    Chunk multiple files in parallel using ProcessPoolExecutor.
 
    Args:
        file_paths:  List of markdown file paths.
        max_workers: Number of parallel processes (defaults to CPU count).
 
    Returns:
        List of chunk dicts from all files in this batch.
    """

    if max_workers is None :
        max_workers = min(os.cpu_count() or 4, len(file_paths))

    all_chunks = []

    with ProcessPoolExecutor(max_workers) as executor:
        results = executor.map(chunk_file, file_paths)
        for file_chunks in results:
            all_chunks.extend(file_chunks)

    return all_chunks




In [ ]:
# Store chunks in ChromaDB
from langchain_chroma import Chroma
from langchain_core.documents import Document
def store_chunks(chunks: list[dict], vector_store: Chroma, project: str):
    """Convert chunk dicts to LangChain Documents and add to chromaDB"""
    documents = []
    ids = []

    for chunk in chunks:
        documents.append(
            Document(
                page_content=chunk["text"],
                metadata = {
                    "source": chunk["source"],
                    "project": project,
                    "heading_trail": chunk["heading_trail"],
                },
            )
        )
        ids.append(chunk["chunk_id"])

    vector_store.add_documents(documents, ids = ids)


In [ ]:
# Clean old chunks before re-ingesting
def clean_project(vector_store: Chroma, project: str):
    """
    Remove all existing chunks for a project before re-ingesting.
    This prevents stale/orphaned chunks when files are edited
    and chunk boundaries shift.
 
    Only deletes chunks for the given project — other projects
    in the same ChromaDB collection are untouched.
    """
    try:
        vector_store._collection.delete(where={"project": project})
        print(f"[Clean] Removed old chunks for project: {project}")
    except Exception:
        # First run — nothing to delete
        print(f"[Clean] No existing data for project: {project}")

In [ ]:

# Full Pipeline
import time
import os
from langchain_openai import OpenAIEmbeddings
def run_pipeline(
    docs_path:str, 
    project:str, 
    batch_size:int = 20, 
    max_workers:int = None, 
    persist_dir: str = "../vectorstore", 
    embedding_model:str = "text-embedding-3-small", 
    fresh:bool = True
):
    """
    Complete ingestion pipeline.
 
    1. Cleans old chunks for this project (if fresh=True)
    2. Collects all .md files
    3. Processes them in batches
    4. Each batch is chunked in parallel (multiprocessing)
    5. Chunks are embedded + stored in ChromaDB
 
    Args:
        docs_path:        Path to documentation directory.
        project:          Project name for metadata filtering.
        batch_size:       Files per batch (controls memory).
        max_workers:      Parallel processes per batch (controls speed).
        persist_dir:      Where ChromaDB stores data on disk.
        embedding_model:  OpenAI embedding model name.
        fresh:            If True, remove old chunks before ingesting.
    """

    start = time.time()

    # Collect
    file_paths = collect_files(docs_path)
    if not file_paths:
        print("No Files Found!")
        return
    
    # Vector Store
    embeddings = OpenAIEmbeddings(model = embedding_model)
    vector_store = Chroma(
        collection_name = project,
        embedding_function=embeddings,
        persist_directory=persist_dir
    )

    # Clean old data when project update 
    if fresh:
        clean_project(vector_store, project)

    # Process in batches
    total_files = len(file_paths)
    total_chunks = 0
    num_batches = (total_files + batch_size - 1) // batch_size

    # For process updates logs
    print(f"\n[Pipeline] {total_files} files, {num_batches} batches of {batch_size}")
    print(f"[Pipeline] Workers: {max_workers or os.cpu_count()}")
    print(f"[Pipeline] Embedding: {embedding_model}\n")


    # Looping through batches
    for i in range(0, total_files, batch_size):            
        num_of_batchs = i // batch_size + 1                 
        batch_file_paths = file_paths[i : i + batch_size]
 
        # Chunk in parallel (multiprocessing)
        chunks = chunk_batchs_in_multiprocess(batch_file_paths, max_workers)
 
        # Store in vector DB
        if chunks:
            store_chunks(chunks, vector_store, project)
            total_chunks += len(chunks)
 
        # Free memory before next batch
        del chunks

        # For process updates logs
        files_done = min(i + batch_size, total_files)
        print(f"  Batch {num_of_batchs}/{num_batches}: "
              f"{files_done}/{total_files} files, "
              f"{total_chunks} chunks so far")
 
    elapsed = time.time() - start
 
    # Summary Status
    print(f"\n{'='*50}")
    print(f"  Project:     {project}")
    print(f"  Files:       {total_files}")
    print(f"  Chunks:      {total_chunks}")
    print(f"  Embedding:   {embedding_model}")
    print(f"  Time:        {elapsed:.1f}s")
    print(f"  Stored in:   {persist_dir}")
    print(f"{'='*50}\n")
 
    return vector_store
 